# Lesson 04 - Tool Use Design Pattern

In this lesson you will learn the **Tool Use** design pattern for AI agents using the Microsoft Agent Framework (Python). We cover:

- Defining function tools with the `@tool` decorator and typed parameters
- Providing tool schemas so the model knows what each tool does
- Controlling tool execution with `approval_mode`
- Returning **structured output** via Pydantic models and `response_format`

The scenario is a **travel booking agent** that can look up destinations, check availability, and retrieve flight information.

## Setup

In [1]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [2]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)
logging.basicConfig(level=logging.DEBUG)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import Message, Agent, tool
# from agent_framework.azure import AzureAIProjectAgentProvider
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework.foundry import FoundryAgent # must need


<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


In [3]:
# Create the Azure AI Foundry provider
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

foundry_client = FoundryChatClient(
  credential=AzureCliCredential(),
  project_endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
  model=os.getenv("FOUNDRY_MODEL")
)

DEBUG:azure.ai.projects.aio._patch:[get_openai_client] Creating OpenAI client using Entra ID authentication, base_url = `https://23120242-2770-resource.services.ai.azure.com/api/projects/23120242-2770/openai/v1`


## Defining Tools with the @tool Decorator

The `@tool` decorator turns a plain Python function into a tool that an agent can call.
Key points:

- The **docstring** becomes the tool description the model sees.
- **Type annotations** (including `Annotated` with descriptions) define the tool schema.
- `approval_mode` controls whether the user must approve each call before it executes.

In [4]:
# Bổ sung để debug xem Agents gọi function calling nào
def log_tool_call(tool_name: str, **kwargs):
    logging.debug(f"[TOOL CALL] {tool_name} | args={kwargs}")

def log_tool_result(tool_name: str, result):
    logging.debug(f"[TOOL RESULT] {tool_name} | result={result}")

In [5]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creating an Agent with Multiple Tools

Pass all three tools to the client so the model can invoke whichever ones it needs to answer the user's question.

In [6]:
travel_tools = [get_destinations, check_availability, get_flight_info]

# agent = await provider.create_agent(
#     name="TravelToolAgent",
#     instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
#     tools=travel_tools,
# )

agent = FoundryAgent(
  name="TravelAvailabilityAgent",
  instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
  agent_name="TravelAvailabilityAgent", #Tên Agent trên Foundry\
 tools=travel_tools,

  credential=AzureCliCredential(),
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

DEBUG:azure.ai.projects.aio._patch:[get_openai_client] Creating OpenAI client using Entra ID authentication, base_url = `https://23120242-2770-resource.services.ai.azure.com/api/projects/23120242-2770/openai/v1`
DEBUG:azure.identity._credentials.azure_cli:Executing subprocess with the following arguments ['C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd', 'account', 'get-access-token', '--output', 'json', '--resource', 'https://ai.azure.com']
INFO:azure.identity._internal.decorators:AzureCliCredential.get_token_info succeeded
DEBUG:azure.identity._internal.decorators:[Authenticated account] Client ID: 04b07795-8ddb-461a-bbee-02f9e1bf7b46. Tenant ID: 40127cd4-45f3-49a3-b05d-315a43a9f033. User Principal Name: 23120242@student.hcmus.edu.vn. Object ID (user): 65b00f45-c4c5-4b43-9c19-affe31aee218
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/responses', 'files': None, 'idempotency_key': 'stainless-python-retry-089ebe19-2f95-4a90-92c3-a01e54df9e36'

I can help you with destinations and availability. Could you please specify if you are interested in certain regions (e.g., Europe, Asia, Americas) or types of destinations (e.g., beach, city, adventure)? This will help me provide you with a relevant list and availability.


## Structured Output with Tools

By setting `response_format` to a Pydantic model, the agent is forced to return a well-typed JSON object instead of free-form text. This is useful when downstream code needs to consume the result programmatically.

In [8]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


# structured_agent = await provider.create_agent(
#     name="StructuredTravelAgent",
#     instructions=(
#         "You are a travel agent. Use the available tools to find destinations, "
#         "check availability, and get flight info. Return structured results."
#     ),
#     tools=[get_destinations, check_availability, get_flight_info],
# )
structured_agent = FoundryAgent(
  name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),  agent_name="TravelAvailabilityAgent", #Tên Agent trên Foundry\
 tools=[get_destinations, check_availability, get_flight_info],

  credential=AzureCliCredential(),
)


response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

DEBUG:azure.ai.projects.aio._patch:[get_openai_client] Creating OpenAI client using Entra ID authentication, base_url = `https://23120242-2770-resource.services.ai.azure.com/api/projects/23120242-2770/openai/v1`
DEBUG:azure.identity._credentials.azure_cli:Executing subprocess with the following arguments ['C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd', 'account', 'get-access-token', '--output', 'json', '--resource', 'https://ai.azure.com']
INFO:azure.identity._internal.decorators:AzureCliCredential.get_token_info succeeded
DEBUG:azure.identity._internal.decorators:[Authenticated account] Client ID: 04b07795-8ddb-461a-bbee-02f9e1bf7b46. Tenant ID: 40127cd4-45f3-49a3-b05d-315a43a9f033. User Principal Name: 23120242@student.hcmus.edu.vn. Object ID (user): 65b00f45-c4c5-4b43-9c19-affe31aee218
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/responses', 'files': None, 'idempotency_key': 'stainless-python-retry-d8566cc1-46ca-440a-b44a-759bc713ee67'

I will check available warm destinations in Europe from London Heathrow. Common warm spots in Europe include Southern Spain, Portugal, Italy, Greece, and the south of France. I will look for flights to these regions. One moment please. Here are some warm destinations in Europe to consider flying from London Heathrow:

1. Malaga, Spain (Costa del Sol) - Known for sunny beaches and resorts.
2. Alicante, Spain - Popular warm coastal city.
3. Faro, Portugal (Algarve region) - Famous for beaches and warm weather.
4. Naples, Italy - Coastal city with warm Mediterranean climate.
5. Athens, Greece - Warm historical city.
6. Nice, France - Warm and sunny city on French Riviera.

I will now check flight availability for these destinations from London Heathrow. One moment please. I have found the following flight options departing from London Heathrow:

1. London Heathrow to Malaga, Spain
   - Airlines: British Airways, easyJet
   - Multiple flights daily
2. London Heathrow to Alicante, Spain
   

## Tool Approval Patterns

The `approval_mode` parameter on `@tool` controls whether tool calls require human approval before executing:

| Mode | Behaviour |
|---|---|
| `"never_require"` | Tool runs automatically — no user confirmation needed. |
| `"always_require"` | Every call must be approved by the user before it executes. |

Use `"always_require"` for tools that have side-effects (e.g. booking a flight, charging a credit card) so a human stays in the loop.

In [9]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

Tool name: book_flight
Approval mode: always_require


## Summary

In this lesson you learned how to:

1. **Define tools** using the `@tool` decorator with typed parameters and docstrings that serve as the tool schema.
2. **Compose multiple tools** so the agent can call them in sequence to answer complex queries.
3. **Return structured output** by passing a Pydantic model as `response_format`.
4. **Control tool approval** with `approval_mode` to keep a human in the loop for sensitive operations.

These patterns form the foundation for building reliable, production-ready agents that can interact with external systems safely.